# 01 — Exploratory Data Analysis

**Goal:** understand the data before modelling. Every finding here should turn into a cleaning rule, a feature idea, or a modelling decision.

> ⚠️ While `source = 'synthetic'`, results describe the generator, not the real market. Re-run this notebook once real data is loaded.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from src.db import read_sql

sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:,.0f}'.format

df = read_sql('SELECT * FROM listings_clean')
df['price_zar'] = df['price_zar'].astype(float)
df['age'] = pd.Timestamp.today().year - df['year']
print(df.shape)
df.head()

## 1. Shape, types and missing values

In [ ]:
df.info()
df.isna().mean().sort_values(ascending=False).to_frame('pct_missing')

## 2. The target: price

Prices are almost always **right-skewed** (lots of cheap cars, few very expensive ones). Compare the raw and log scale.
Question to answer: *should the model predict `price` or `log(price)`?*

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['price_zar'], bins=60, ax=ax[0]).set_title('Price (R)')
sns.histplot(np.log10(df['price_zar']), bins=60, ax=ax[1]).set_title('log10(Price)')
plt.show()
df['price_zar'].describe()

## 3. Depreciation: price vs age

✍️ **Your notes:** Is the decline linear or exponential? Which make loses value fastest?

In [ ]:
top_makes = df['make'].value_counts().head(6).index
curve = (df[df['make'].isin(top_makes)]
         .groupby(['make', 'age'])['price_zar'].median().reset_index())
plt.figure(figsize=(10, 5))
sns.lineplot(data=curve, x='age', y='price_zar', hue='make', marker='o')
plt.yscale('log'); plt.title('Median price by age (log scale)'); plt.show()

### Why is the Toyota line bumpy?

Toyota's mix of models (Yaris vs Fortuner) differs at each age, so the *make-level* median jumps around.
This is a **composition effect** — a key reason the model needs `model`, not just `make`. Check it per model:

In [ ]:
toy = df[df['make'] == 'Toyota'].groupby(['model', 'age'])['price_zar'].median().reset_index()
plt.figure(figsize=(10, 5))
sns.lineplot(data=toy, x='age', y='price_zar', hue='model', marker='o')
plt.yscale('log'); plt.title('Toyota by model'); plt.show()

## 4. Mileage vs price

In [ ]:
sns.scatterplot(data=df.sample(min(2000, len(df)), random_state=0),
                x='mileage_km', y='price_zar', hue='age', palette='viridis', alpha=0.6, s=15)
plt.yscale('log'); plt.title('Mileage vs price, coloured by age'); plt.show()
print('Correlation (Spearman):')
df[['price_zar', 'age', 'mileage_km']].corr(method='spearman')

Age and mileage are strongly correlated with each other. **Feature idea:** `km_per_year = mileage / max(age, 1)` separates *how much* a car was driven from *how old* it is.

## 5. Categorical effects: fuel, transmission, seller, province

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, ['fuel_type', 'transmission', 'seller_type', 'province']):
    order = df.groupby(col)['price_zar'].median().sort_values().index
    sns.boxplot(data=df, y=col, x='price_zar', order=order, ax=ax, showfliers=False)
    ax.set_xscale('log'); ax.set_title(col)
plt.tight_layout(); plt.show()

⚠️ **Raw comparisons lie.** Diesel cars look pricier partly because bakkies and SUVs are diesel.
To isolate the diesel effect, compare *within the same make/model/year* (see query 4 in `sql/eda_queries.sql`) — or let the model control for it.

## 6. Findings → next steps

✍️ Fill in as you go:

| Finding | Action |
|---|---|
| Price is right-skewed | Try predicting log(price) |
| Composition effects across models | Use model (and variant) as features |
| Age & mileage correlated | Engineer km_per_year |
| … | … |

**Next:** `02_features.ipynb` → baseline linear regression.